In [ ]:
import numpy as np
import xarray as xr
import json
import os

import matplotlib.pyplot as plt
from matplotlib import colors
from scipy.optimize import curve_fit
from matplotlib.colors import LogNorm

from feature_retreiver import *
from utility import (
    compute_tke_bl, surface_value_z_grid, last_profile, compute_bl_rh18
)

## SINGLE TRAINING SET ANALYSIS

In [ ]:
training_set_name = "ePBL_paper_2283_corrected"
root_dir = "dataset_maker_for_gotm"

In [ ]:
with open(f"{root_dir}/case_dictionaries/{training_set_name}_training_set_cases.json", "r") as file:
    gotm_case_dict = json.load(file)

In [ ]:
feature_retreiver = FeatureRetreiver(
    training_set_dir=os.path.join(root_dir, f"{training_set_name}_training_dataset"),
    case_dict=gotm_case_dict,
    grid="constant",
    dz=1,
    dt=1800
)

In [ ]:
bl_dataset = feature_retreiver.make_dataset_from_processed_data(
    processing_method=compute_bl_rh18,
    processing_method_name="bl",
    variable="nuh",
    coordinates=["time"]
)

In [ ]:
tke_bl_dataset = feature_retreiver.make_dataset_from_processed_data(
    processing_method=compute_tke_bl,
    processing_method_name="bl",
    variable="tke",
    coordinates=["time"]
)

In [ ]:
Rig_bl_dataset = feature_retreiver.make_var_at_bl_dataset(variable="Rig", z_max=400, index_above_bl=0)

In [ ]:
u_dataset = feature_retreiver.make_dataset_from_processed_data(
    processing_method=surface_value_z_grid,
    processing_method_name="u_surf",
    variable="u",
    coordinates=["time"]
)

## CASE COMPARISON

In [ ]:
root_dir = "."

training_set_name_1 = "ePBL_paper_GLS"
training_set_name_2 = "ePBL_paper_2283_corrected"

In [ ]:
with open(f"{root_dir}/{training_set_name_1}_training_set_cases.json", "r") as file:
    gotm_case_dict_1 = json.load(file)

with open(f"{root_dir}/{training_set_name_2}_training_set_cases.json", "r") as file:
    gotm_case_dict_2 = json.load(file)

In [ ]:
target_specs = {'temp_grad': 0.001, 'tx': 0.1, 'lat': 10.0, 'heat_flux': -100}

for case, specs in gotm_case_dict_1.items():
    if specs == target_specs:
        case_1 = case 
    else: pass

for case, specs in gotm_case_dict_2.items():
    if specs == target_specs:
        case_2 = case
    else: pass

In [ ]:
ds_1 = xr.open_dataset(f"{root_dir}/{training_set_name_1}_training_dataset/{case_1}/output.nc").isel(lat=0, lon=0)
ds_2 = xr.open_dataset(f"{root_dir}/{training_set_name_2}_training_dataset/{case_2}/output.nc").isel(lat=0, lon=0)